# MIMIC-IV 30-Day Readmission — Data Representation

Development notebook for the production MLOps pipeline on GCP.

**Scope of this notebook:** build the BigQuery data representation only —
cohort, label, feature views, and clinical baselines. Modeling and Vertex AI
orchestration live in separate artifacts.

**Working principle:** one cell at a time. Each cell has a single, named
purpose. Open design questions are tracked in
[`docs/open_questions.md`](docs/open_questions.md).

## Cell 1 — Setup & configuration

Single source of truth for project, datasets, and the BigQuery client.
No queries are executed here.

In [1]:
from google.cloud import bigquery

# --- Project & location -------------------------------------------------
PROJECT_ID = "enterprise-clinical-copilot"
LOCATION = "US"

# --- MIMIC-IV source (PhysioNet public BigQuery) ------------------------
# Datasets confirmed available in this GCP project:
#   physionet-data.mimiciv_3_1_hosp     - core hospital tables (versioned)
#   physionet-data.mimiciv_3_1_icu      - ICU tables           (versioned)
#   physionet-data.mimiciv_3_1_derived  - mimic-code concepts  (versioned)
#   physionet-data.mimiciv_ed           - ED module            (unversioned)
#   physionet-data.mimiciv_note         - clinical notes       (unversioned)
MIMIC_VERSION = "v3_1"

SOURCE_PROJECT = "physionet-data"
SOURCE_HOSP = f"{SOURCE_PROJECT}.mimiciv_3_1_hosp"
SOURCE_ICU = f"{SOURCE_PROJECT}.mimiciv_3_1_icu"
SOURCE_DERIVED = f"{SOURCE_PROJECT}.mimiciv_3_1_derived"
SOURCE_ED = f"{SOURCE_PROJECT}.mimiciv_ed"
SOURCE_NOTE = f"{SOURCE_PROJECT}.mimiciv_note"

# --- Curated destination ------------------------------------------------
DEST_DATASET = "readmission"
DEST = f"{PROJECT_ID}.{DEST_DATASET}"

# --- BigQuery client ----------------------------------------------------
bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print(f"Project:       {PROJECT_ID}")
print(f"Location:      {LOCATION}")
print(f"MIMIC hosp:    {SOURCE_HOSP}")
print(f"MIMIC icu:     {SOURCE_ICU}")
print(f"MIMIC derived: {SOURCE_DERIVED}")
print(f"MIMIC ed:      {SOURCE_ED}")
print(f"MIMIC note:    {SOURCE_NOTE}")
print(f"Curated dest:  {DEST}")


Project:       enterprise-clinical-copilot
Location:      US
MIMIC hosp:    physionet-data.mimiciv_3_1_hosp
MIMIC icu:     physionet-data.mimiciv_3_1_icu
MIMIC derived: physionet-data.mimiciv_3_1_derived
MIMIC ed:      physionet-data.mimiciv_ed
MIMIC note:    physionet-data.mimiciv_note
Curated dest:  enterprise-clinical-copilot.readmission


## Part 1 — Patient cohort (index admissions)

Builds `readmission.cohort_index_admissions` (one row per patient).

Logic, in order, per the resolved decisions:

1. **Adult**: `anchor_age + (YEAR(admittime) - anchor_year) >= 18`.
2. **LOS ≥ 24h**: `TIMESTAMP_DIFF(dischtime, admittime, HOUR) >= 24`.
3. **Mortality exclusion**: `hospital_expire_flag = 0 AND deathtime IS NULL`.
4. **Discharge disposition exclusion**: `discharge_location` not in
   `HOSPICE`, `AGAINST ADVICE`, `OTHER FACILITY`, `ACUTE HOSPITAL`.
5. **Filter-first → rank**: among surviving admissions, take the earliest
   `admittime` per `subject_id` as the index.

`admission_type` is **not** filtered here — it only affects the label
(qualifying readmission triggers in Part 2).


In [2]:
# Cell 2 — Ensure destination dataset exists (idempotent, one-time).
from google.cloud.exceptions import NotFound

_dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DEST_DATASET}")
_dataset_ref.location = LOCATION
_dataset_ref.description = "Curated tables for MIMIC-IV 30-day readmission pipeline."

try:
    bq.get_dataset(_dataset_ref)
    print(f"Dataset already exists: {DEST}")
except NotFound:
    bq.create_dataset(_dataset_ref)
    print(f"Created dataset: {DEST}")


Created dataset: enterprise-clinical-copilot.readmission


In [4]:
# Cell 3 — Build the cohort table (Part 1, steps 1-5).
#
# Note on physical layout: MIMIC-IV randomly shifts each patient's dates
# across a ~100-year span, so DAY-level partitioning on admittime exceeds
# BigQuery's 4,000-partition limit. The cohort is also small (~one row per
# patient). We therefore skip partitioning and rely on clustering by
# subject_id, which is the join key for every downstream feature view.
COHORT_TABLE = f"{DEST}.cohort_index_admissions"

cohort_sql = f"""
CREATE OR REPLACE TABLE `{COHORT_TABLE}`
CLUSTER BY subject_id AS
WITH base AS (
  SELECT
    a.subject_id,
    a.hadm_id,
    a.admittime,
    a.dischtime,
    a.admission_type,
    a.admission_location,
    a.discharge_location,
    a.insurance,
    a.language,
    a.marital_status,
    a.race,
    a.edregtime,
    a.edouttime,
    p.gender,
    p.anchor_age,
    p.anchor_year,
    p.anchor_year_group,
    p.anchor_age + (EXTRACT(YEAR FROM a.admittime) - p.anchor_year) AS age_at_admission,
    TIMESTAMP_DIFF(a.dischtime, a.admittime, HOUR) AS los_hours
  FROM `{SOURCE_HOSP}.admissions` a
  JOIN `{SOURCE_HOSP}.patients`   p USING (subject_id)
  WHERE a.admittime IS NOT NULL
    AND a.dischtime IS NOT NULL
    -- Step 3: mortality exclusion
    AND a.hospital_expire_flag = 0
    AND a.deathtime IS NULL
    -- Step 4: discharge disposition exclusion (NULLs retained)
    AND (
      a.discharge_location IS NULL
      OR a.discharge_location NOT IN (
        'HOSPICE', 'AGAINST ADVICE', 'OTHER FACILITY', 'ACUTE HOSPITAL'
      )
    )
),
eligible AS (
  SELECT *
  FROM base
  WHERE age_at_admission >= 18    -- Step 1
    AND los_hours          >= 24  -- Step 2
),
ranked AS (
  -- Step 5: filter-first then rank; earliest surviving admission per patient.
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY admittime, hadm_id) AS admit_rank
  FROM eligible
)
SELECT * EXCEPT (admit_rank)
FROM ranked
WHERE admit_rank = 1
"""

job = bq.query(cohort_sql)
job.result()  # block until complete
table = bq.get_table(COHORT_TABLE)
print(f"Built {COHORT_TABLE}")
print(f"  rows:           {table.num_rows:,}")
print(f"  bytes:          {table.num_bytes / 1e6:,.1f} MB")
print(f"  cluster col:    subject_id")


Built enterprise-clinical-copilot.readmission.cohort_index_admissions
  rows:           179,224
  bytes:          30.9 MB
  cluster col:    subject_id


## Part 2 — Target label (30-day unplanned readmission)

Builds `readmission.cohort_labeled` (one row per patient = cohort + label).

For each index admission, scan the raw `admissions` table for the same
`subject_id` and apply, in order:

1. **Subsequent**: `next.admittime > index.dischtime`.
2. **30-day window**: `DATE_DIFF(next.admittime, index.dischtime, DAY) BETWEEN 1 AND 30`.
3. **Acute trigger**: `next.admission_type IN (
   'URGENT','EMERGENCY','EW EMER.','DIRECT EMER.',
   'DIRECT OBSERVATION','EU OBSERVATION','OBSERVATION ADMIT',
   'AMBULATORY OBSERVATION')`.
   `'ELECTIVE'` and `'SURGICAL SAME DAY ADMISSION'` are excluded as triggers.
4. **Assignment**: `label = 1` iff *any* subsequent admission meets all three;
   otherwise `label = 0` (no return, return > 30 days, or in-window returns
   are all elective/planned).

Diagnostic columns are emitted for auditability:
`days_to_readmit`, `next_admission_type`, `n_readmits_30d_any`.



In [5]:
# Cell 4 — Build the labeled cohort (Part 2, steps 1-4).
LABELED_TABLE = f"{DEST}.cohort_labeled"

ACUTE_TYPES = (
    "'URGENT'",
    "'EMERGENCY'",
    "'EW EMER.'",
    "'DIRECT EMER.'",
    "'DIRECT OBSERVATION'",
    "'EU OBSERVATION'",
    "'OBSERVATION ADMIT'",
    "'AMBULATORY OBSERVATION'",
)
acute_types_sql = ", ".join(ACUTE_TYPES)

label_sql = f"""
CREATE OR REPLACE TABLE `{LABELED_TABLE}`
CLUSTER BY subject_id AS
WITH subsequent AS (
  -- Step 1+2: for each index, find later admissions within 30 days.
  -- Step 3: acute flag on each candidate.
  SELECT
    c.subject_id,
    c.hadm_id                                                      AS index_hadm_id,
    DATE_DIFF(DATE(n.admittime), DATE(c.dischtime), DAY)           AS days_to_readmit,
    n.admission_type                                               AS next_admission_type,
    n.admission_type IN ({acute_types_sql})                        AS is_acute
  FROM `{COHORT_TABLE}`              c
  JOIN `{SOURCE_HOSP}.admissions`    n
    ON n.subject_id = c.subject_id
   AND n.hadm_id   != c.hadm_id
   AND n.admittime  > c.dischtime
   AND DATE_DIFF(DATE(n.admittime), DATE(c.dischtime), DAY) BETWEEN 1 AND 30
),
agg AS (
  -- Step 4: collapse to one row per index admission.
  -- Pick the EARLIEST qualifying acute readmission for the diagnostic columns.
  SELECT
    subject_id,
    index_hadm_id,
    COUNTIF(TRUE)                                                  AS n_readmits_30d_any,
    LOGICAL_OR(is_acute)                                           AS has_acute_30d,
    MIN(IF(is_acute, days_to_readmit, NULL))                       AS days_to_readmit,
    ARRAY_AGG(
      IF(is_acute, next_admission_type, NULL) IGNORE NULLS
      ORDER BY days_to_readmit
      LIMIT 1
    )[SAFE_OFFSET(0)]                                              AS next_admission_type
  FROM subsequent
  GROUP BY subject_id, index_hadm_id
)
SELECT
  c.*,
  CAST(COALESCE(a.has_acute_30d, FALSE) AS INT64) AS label,
  COALESCE(a.n_readmits_30d_any, 0)               AS n_readmits_30d_any,
  a.days_to_readmit,
  a.next_admission_type
FROM `{COHORT_TABLE}` c
LEFT JOIN agg a
  ON a.subject_id    = c.subject_id
 AND a.index_hadm_id = c.hadm_id
"""

bq.query(label_sql).result()

# --- Sanity-check the label distribution --------------------------------
stats_sql = f"""
SELECT
  COUNT(*)                                                        AS n_total,
  COUNTIF(label = 1)                                              AS n_positive,
  ROUND(SAFE_DIVIDE(COUNTIF(label = 1), COUNT(*)) * 100, 2)       AS positive_rate_pct,
  COUNTIF(n_readmits_30d_any > 0 AND label = 0)                   AS n_30d_elective_only,
  AVG(IF(label = 1, days_to_readmit, NULL))                       AS mean_days_to_readmit
FROM `{LABELED_TABLE}`
"""
stats = bq.query(stats_sql).result().to_dataframe().iloc[0]

table = bq.get_table(LABELED_TABLE)
print(f"Built {LABELED_TABLE}")
print(f"  rows:                    {table.num_rows:,}")
print(f"  positives (y=1):         {int(stats.n_positive):,}")
print(f"  prevalence:              {stats.positive_rate_pct:.2f}%")
print(f"  in-window elective-only: {int(stats.n_30d_elective_only):,}  (kept as y=0)")
print(f"  mean days to readmit:    {stats.mean_days_to_readmit:.1f}")


Built enterprise-clinical-copilot.readmission.cohort_labeled
  rows:                    179,224
  positives (y=1):         21,736
  prevalence:              12.13%
  in-window elective-only: 2,204  (kept as y=0)
  mean days to readmit:    11.2
